# Teaching Models to Check Their Own Work: Four Approaches to Verification
*From hand-labeled outcome verifiers, to step-by-step process rewards, to label-free self-checking, to combining many weak verifiers into one strong one*

# Why We Need Verification
We've seen that language models can often produce a correct answer if you let them try many times (repeated sampling). But there's a catch: **how do we automatically pick the correct answer out of all the ones it generated?**

This is the problem of **verification** — building a way to check which answer is right, or to guide the model while it's generating an answer. This notebook covers four papers that show how people's approach to verification has changed over time. We start with the earliest one.


# Paper 1: Training Verifiers to Solve Math Problems (OpenAI, 2021)
**Paper:** [arxiv.org/abs/2110.14168](https://arxiv.org/abs/2110.14168)

The starting problem: language models **hallucinate** — meaning they can give a wrong answer while sounding completely confident about it. This is still true today, even though models have gotten much better since 2021.

## A New Math Benchmark: GSM8K
This paper also introduced a new dataset of math problems, called **GSM8K**. It's still widely used today, especially for testing smaller models.

- **8,500 grade-school math problems.**
- Problems are simple on the surface, but need a **few steps of reasoning** to solve — not just one calculation.
- Solutions are written out in **natural language**, not just raw math.


# What Is a Verifier?
A verifier is a model that looks at a question and a proposed solution, and outputs the **probability that the solution is correct**.

Think of it like a grading rubric: humans solving problems often benefit from a way to check whether they got it right. A verifier gives a language model something similar — a second model that checks its work.


# How the Verifier Was Trained

## Step 1: Create Training Data
1. Take a question with a known, human-verified correct answer.
2. Let the generator model (the LLM answering questions) produce **100 different solutions** to that question (repeated sampling).
3. Compare each of the 100 solutions to the true answer, and label each one **correct or incorrect**.

Since the humans only had to solve each problem once to get the ground truth, and the labeling of the 100 generated solutions was done automatically by comparing to that ground truth, this whole process didn't need 100x the human effort — just one correct answer per problem, done offline in advance.

## Step 2: Train the Verifier
The verifier is trained on these (question, solution, label) triples. Its job: predict whether a solution is correct.

## Step 3: Use It at Test Time
At test time, generate many candidate answers again, run the verifier on each one, and pick the answer with the **highest score** as the final output.


# How the Verifier Model Is Built
The verifier is itself a language model, with one small addition: a **scalar head** that outputs a correct/incorrect score for **each token**, not just once for the whole answer.

- The question tokens are **masked out** — the loss doesn't apply to them, only to the solution tokens.
- The verifier is trained with **two losses combined**:
  1. A **binary loss** — is this solution correct or not?
  2. A normal **language modeling loss** — predict the next token, same as regular LLM training.

The paper found training worked better with both losses combined, rather than just the binary correctness loss alone.

## Training Recipe Used
1. Fine-tune the generator model for 2 epochs.
2. Sample 100 completions per question from the fine-tuned generator.
3. Label each completion correct/incorrect (by comparing to the known right answer).
4. Train the verifier for 1 epoch on this labeled data.

Note: the initial fine-tuning step (step 1) may not even be necessary with today's stronger base models — many newer verifiers skip straight to verifier training without it.


# Sentence-Level vs. Token-Level Labels
The paper tried two ways of labeling correctness:

- **Sentence-level:** after each sentence (roughly, after each period), judge whether that step was correct or not.
- **Token-level:** label every single token — a much noisier signal, since we want the whole sequence of tokens to lead toward a correct final answer.

## Turning Per-Token Scores Into One Final Answer
Since a real decision needs just one yes/no per full solution (not per token), the paper used a simple rule: **look at the score of the very last token**. If the last token's score says "correct," the whole solution is treated as correct.

In practice, this worked visibly well: when they colored each token green (high score) or red (low score) along a solution, correct solutions tended to trend from red/mixed early on toward green by the end, and the final (last-token) score usually matched whether the answer was actually right.


# Does Verification Actually Help?
The paper compared two approaches:

- **Baseline:** just fine-tune the generator model directly (plain supervised fine-tuning, next-token prediction).
- **Verification:** generate 100 samples per question, then use the verifier to pick the best one.

## Result
For both a 6B and a 175B model (GPT-3-sized), verification **beat plain fine-tuning** — but only once the verifier had enough training data. With a small training set (under ~1,000 examples for the 175B model), verification didn't help much yet. As the verifier's training set grew, it started to clearly outperform the fine-tuning-only approach.


# Generator Size vs. Verifier Size
An interesting side result, worth thinking about as a possible research project: what happens if you mix a big generator with a small verifier, versus a small generator with a big verifier?

**Finding:** a **big generator + small verifier** worked better than the reverse.

This makes some intuitive sense — generating a correct answer is generally a harder task than checking whether a given answer is correct, so it's reasonable that verification needs less model capacity than generation does. Finding the best trade-off between generator size and verifier size is still an open question, especially now that both base generator models and available verifier models (there are leaderboards of these on Hugging Face) have improved a lot since 2021.


# How Many Samples Should You Generate?
As the paper increased the number of completions sampled per problem, accuracy kept improving up to around **400 samples** — after that, more samples didn't help, and even slightly hurt.

## Why It Levels Off (and Even Drops)
This isn't the same measurement as "coverage" (whether at least one of the samples is correct) — it's the accuracy of the verifier's **top pick** among all the samples. As the number of candidates grows very large, some solutions become very close to each other in quality, and the verifier struggles to tell a slightly-wrong one from the truly correct one. That's why accuracy can actually dip after the peak, rather than just flattening out.

**In practice**, the paper's final system used only 100 samples, since most of the benefit was already captured there and results were more consistent.

## Comparing to Majority Voting
Recall: plain majority voting (picking the most common answer) stopped improving after only about 10–50 samples. This verifier-based approach kept improving much further — up to about 400 samples — showing that a trained verifier captures a lot more signal than simple vote-counting.


# Does It Make Sense to Scale the Verifier Too?
If scaling how many answers you generate helps, does scaling the verifier itself (e.g., running it with more test-time compute) also help? Yes — and this idea shows up in later verification research.


# Why Use a Verifier Instead of Just Fine-Tuning More?
If you have a huge amount of training data, fine-tuning and verification may end up performing about the same. But a verifier has one big advantage: it **doesn't change the base generator model**.

This means the generator stays general-purpose — it isn't overfit or specialized to one narrow dataset or task. The verifier becomes a separate, add-on tool that can guide the same general base model toward better answers, without sacrificing its general ability.


# Paper 2: Let's Verify Step by Step (OpenAI, ~2 years later)
**Paper:** [arxiv.org/abs/2305.20050](https://arxiv.org/abs/2305.20050)

Same core problem as before: LLMs still hallucinate, and if they make one wrong move early in a multi-step solution, that single mistake can throw off the entire final answer.

This paper compares two ways of building a reward model (a model that scores whether an answer is good or bad):
- **Outcome-based reward model (ORM):** gives one score for the whole solution — like Paper 1.
- **Process-based reward model (PRM):** gives a separate score for **each individual step** of the solution, not just the final answer.

Their final process-supervised model solved **78%** of problems on a tricky subset of the MATH test set (a much harder dataset than GSM8K) — a strong result at the time.


# How ORM and PRM Are Labeled Differently
Take one math problem. The generator produces one solution, broken into steps (step 1, step 2, ... final answer).

- **ORM:** just check the final answer against the known correct answer. One label for the whole thing.
- **PRM:** have human annotators look at **every step** and label each one correct or incorrect. If all the steps are marked correct, that's usually why the final answer came out right too.

To turn a PRM's per-step scores into one overall solution score, the paper multiplies the step scores together — this is called **process supervision**.


# Why Process Supervision Helps
The biggest advantage: process supervision cuts down on **false positives**.

Here's the problem it solves — sometimes a model can accidentally hallucinate its way to the *correct final answer* even though the actual reasoning steps along the way were wrong. An outcome-based reward model would score this as fully correct, since it only checks the final answer. A process-based reward model is much less likely to be fooled this way, because it's checking each step, not just the ending.

As a side benefit, this also pushes the model toward reasoning that's easier for humans to follow and check, since every step was reviewed and labeled by a human.


# Building the PRM800K Dataset
The paper built and released **PRM800K** — an open dataset of 800,000 step-level human labels.

## How They Collected Labels Efficiently
Rather than randomly picking generated solutions to label, they prioritized **"convincing wrong" answers** — cases where the final answer looked correct, but one or more of the steps in between were actually wrong. Focusing human labeling effort on these tricky cases made the whole process **2.6x more data-efficient** than labeling random samples.

They also improved the PRM iteratively: collect labels → train the PRM → use the improved PRM to help pick better/harder samples to label next → repeat.

## Example: How a Step Gets Labeled
For a sample math problem, steps 1 through 5 might all be labeled **correct** — but if the model makes an arithmetic mistake in the final step (like getting x = 7 instead of x = 14), that last step gets labeled **incorrect**, even though everything before it was fine. Each step is labeled positive, negative, or neutral.

## Training Setup
Both ORM and PRM were trained starting from a fine-tuned **GPT-4** base:
- **ORM:** labeled pairs of (generated solution, correct or not). Final score = the score on the last token, same approach as Paper 1.
- **PRM:** labeled (generated step, correct or not) pairs. Final score = the **product of the per-step scores**.


# Results: PRM Beats ORM and Majority Voting
Testing on the same kind of "n samples per problem" setup:

- **Majority voting** plateaus early, around 100 samples — same limitation seen before.
- **ORM** does better than majority voting, but PRM does even better.
- **PRM's biggest edge:** it's much better at correctly identifying rare correct answers — problems where less than 5% of the sampled solutions happen to be right.

## PRM Is More Data-Efficient
Comparing labels needed: getting the same benefit from ORM might take, say, 100 labeled solutions per problem, while PRM can get similar or better results with far fewer full solutions (since each solution gives many step-level labels, not just one).

## PRM Generalizes Better
On new domains/datasets outside what it was trained on, PRM handled the shift much better than either ORM or majority voting — even though majority voting actually beat ORM in this generalization test, PRM still came out on top overall.


# Where PRM Can Go Wrong

## Can a PRM Give a High Score to a "Lazy" or Skipped Reasoning Step?
What if a model skips the real reasoning and just writes something like *"let numerator = x, so x = 14"* — jumping straight to a plausible-looking answer? Would the PRM catch that this step was hollow?

**It depends on how it's used.** If you're just using the PRM to **score and pick between different generator outputs**, the generator itself is untouched — so you can still prompt the generator to show its full reasoning step by step, and the PRM simply scores whatever steps it's given. The risk shows up if you go further and use the PRM's score to **fine-tune the generator itself** — in that case, the generator could learn to shortcut straight to answers the PRM happens to like, skipping real reasoning. Careful PRM design (and training it on genuinely well-reasoned human-labeled steps, so a "shortcut" step gets a low score) helps guard against this — but it's a real risk to watch for, especially when the PRM's own labels aren't human-made (a caveat covered further in the next paper).

## Does PRM Ever Hurt?
Yes — a PRM's per-step judgment can sometimes be wrong (assigning "credit" to a step that looks fine but doesn't actually help reach the answer). In practice, many newer systems **combine PRM and ORM together** to get the benefits of both. Using a PRM also adds a new setting to tune: the threshold for what counts as a "good enough" step score, which adds complexity to the system.

## Is the ORM-vs-PRM Comparison Fair?
PRM needs far more labels than ORM (roughly: one label per step, versus one label per solution) — so comparing them on the same x-axis (number of solutions) might not be a fair comparison of "cost." This is a fair concern — it's genuinely hard to control for exactly, since the number of steps varies per problem, but the paper does try to account for this in some of its charts.

## Clarifying "Majority Voting"
Just to be clear: majority voting only looks at the **final answer**, not the reasoning — it picks whichever final answer shows up most often across the samples, with no reward model involved at all.


# Paper 3: Math-Shepherd — Verifying Without Human Labels
**Paper:** [arxiv.org/abs/2312.08935](https://arxiv.org/abs/2312.08935)

The last paper needed a lot of human-labeled data to train its PRM. That takes a lot of time and money. This paper asks a simple question: **can we build a PRM without any human labeling at all?**

## The Core Idea
Define a step's quality as: **how likely is this step to lead to the correct final answer?** To measure this, from any given step, sample N different ways to continue, and see how many of them end up correct.

Two ways to turn that into a score:
- **Hard estimate:** did *any* of the N continuations reach the correct final answer? (Yes/no.)
- **Soft estimate:** *what fraction* of the N continuations reached the correct final answer? (A number between 0 and 1.)

**Example:** from a given step, sample 3 possible continuations. If 2 out of 3 reach the correct answer, the soft estimate score is 2/3, and the hard estimate score is 1 (since at least one succeeded).


# Weaknesses of This Auto-Labeling Approach
Two real problems with this method:

- **Unusual but valid solution paths can get scored unfairly low.** If a step leads to a correct answer through an uncommon path, but you only sample a small number of continuations (like N=3), you might miss the successful path entirely and wrongly score that step as bad. A larger N (like 100) might have caught it.
- **Hard problems give little to no signal.** If a problem is genuinely hard, most sampled continuations — even from a good step — may fail to reach the correct answer. This makes it hard to tell a good step from a bad one on hard problems.
- **A related risk:** if a wrong step happens to still lead to the correct final answer (by luck), it can get mislabeled as "correct" too. More sampling might reveal that this path is unreliable, but there's no guarantee.


# How Math-Shepherd Uses the Verifier
At test time: sample many candidate full solutions, score them with the trained PRM, and pick the one with the highest score.

They also went a step further: they used this PRM as a **reward signal to train the generator itself** with reinforcement learning — pushing the generator to naturally produce steps the PRM scores highly.

## Hard vs. Soft Estimate — Does It Matter?
Soft estimates looked slightly better in general, but the real gains came from **N=4** (how many continuations to sample per step to build the label) — the exact choice of hard vs. soft mattered less than getting this sampling depth right. They ended up using the hard estimate anyway, mainly because it's simpler to compute.


# Results: Math-Shepherd vs. Other Methods
Comparing several approaches:
- **Self-consistency (SC):** another name for majority voting — pick the most repeated final answer.
- **ORM**
- **Math-Shepherd (their PRM approach, with no human labels at all)**

**Result:** Math-Shepherd beat both self-consistency and ORM — and this was achieved with **zero human annotation**. It also beat PRM800K (the human-labeled PRM from the previous paper) on the harder MATH dataset, with an even bigger gap there than on GSM8K.

This pattern held consistently across different base models tested (LLaMA-70B, LLema-34B, DeepSeek-67B) — Math-Shepherd's approach clearly beat plain self-consistency in each case.

**Concrete numbers:** with Math-Shepherd verification, DeepSeek-67B reached **93.3% on GSM8K and 48.1% on MATH**. Separately, applying Math-Shepherd's PPO-based RL training (covered next) took Mistral-7B from 77.9%→84.1% on GSM8K and 28.6%→33.0% on MATH — and combining that RL training with verification on top pushed it even further, to 89.1% and 43.5%.

Interestingly, Math-Shepherd's fully **automatic** labels even outperformed the **human-labeled** PRM800K on the MATH dataset — likely because Math-Shepherd generated roughly 4x more labeled data and that data better matched the specific open-source models being tested.


# Using the PRM to Improve the Generator via RL
Beyond just picking the best answer at test time, they also used the PRM as a reward signal to fine-tune the generator directly, using **PPO** (a reinforcement learning method) — tested on Mistral-7B.

- Training the generator against the **PRM** reward worked better than training against an **ORM** reward.
- The improvement from this RL step was smaller than what pure test-time scaling (sampling + verifying) achieved on its own — but it's a genuinely different, complementary way to improve the model.
- Gains from this RL approach appeared to **plateau** after a point.

**Big picture:** this is a fully self-improving loop — the model generates its own training signal (no humans needed), a PRM is trained on that signal, and the PRM is then used both (1) at test time to pick better answers, and (2) as a reward to directly fine-tune the generator via RL.


# More Open Questions

## Can PRMs Be Trained to Reward Self-Correction?
Since we're only scoring whether each step is correct, are we missing a chance to reward the model for **catching and fixing its own mistakes**? One suggested method is to score steps against a **rubric** (a defined checklist of what a good step should look like), which can include things like whether the model used a tool (a calculator, or a symbolic math library like SymPy) to solve a piece of the problem — giving extra signal beyond a simple right/wrong per step.

## Is Compute Being Compared Fairly Between ORM and PRM?
ORM and PRM use compute very differently (PRM needs many more generated samples per problem to build its step-level scores), so comparing them directly on things like "number of samples" might not reflect a truly fair, equal-compute comparison. This is a real challenge — ORM and PRM should ideally be trained on the exact same underlying data to make for a fair comparison, since the training data itself has a big effect on results, and controlling for total compute across such different training approaches is genuinely difficult. This remains a good open research question.

## Why Stick with Greedy Decoding After RL Training?
After RL-training the generator, why evaluate it with plain greedy decoding rather than also trying temperature-based sampling and further test-time scaling on top? This wasn't explored in the paper, and would be an interesting follow-up — for example, repeating the whole labeling and PRM-training process again, but starting from the already RL-improved model.


# Paper 4: Weaver — Shrinking the Gap with Weak Verifiers
**Paper:** [arxiv.org/abs/2506.18203](https://arxiv.org/abs/2506.18203)

This is a newer paper. It has the same goal as before — close the gap between what a model can generate and what a verifier can actually pick out — but takes a very different approach: **instead of training one new verifier from scratch, combine several existing, imperfect verifiers into one much stronger system.**

## Why "Weak" Verifiers?
No single verifier is perfect — that's what "weak" means here. These aren't bad verifiers on purpose; they're simply the best verifiers currently available, each with its own imperfections. The idea is to combine several of them (an ensemble) to get something more accurate than any single one alone.

## Types of Verifiers in the Pool
- **PRMs and ORMs** (from earlier papers).
- **LLM-as-judge:** just ask an LLM directly, "do you think this answer is correct?" — optionally giving it tools or a rubric to help it decide.


# Does Simple Ensembling Already Help?
Just combining multiple verifiers (top 1, top 5, top 10, ranked by quality) already improves results, though not perfectly smoothly as you add more.

**What consistently helped:** learning a **weight for each verifier** using a small labeled dataset, rather than treating every verifier equally. Two simple ways to do this: **Naive Bayes** or **logistic regression** — both just assign one weight per verifier and combine their scores using those weights, learned from a training set, then applied to new (test) data.


# How Weaver Works: Score, Weight, Select
Weaver's process has three steps:

1. **Score and normalize:** collect scores from all the verifiers, put them on the same scale, and **filter out low-quality verifiers** entirely (based on a small labeled sample). This filtering step turned out to matter a lot — a verifier needs to clear a quality bar just to be included in the ensemble.
2. **Weight:** use "weak supervision" to estimate how accurate each verifier is, using only a **small amount** of labeled data.
3. **Select:** combine all the verifiers' scores using those learned weights, into one final score.

## The Weak-Supervision Idea (Inspired by Snorkel)
This approach — combining several imperfect ("weak") signals into one stronger signal — comes from earlier work on weak supervision (notably Snorkel, by Alex Ratner and others).

**Setup:** you have $n$ questions, $k$ generated solutions per question, and $m$ verifiers — giving $n \times k \times m$ total labels. The goal: estimate the probability that a given solution is actually correct, using all these (partly unreliable) verifier opinions.

**Key assumption:** each verifier captures somewhat **independent** information about correctness. Think of it like asking several different judges to review the same answer — if all the judges always agree with each other, you're not really learning anything new by asking more of them. The real value comes from where judges *agree and disagree* with one another. Under this assumption, the probability that an answer is correct can be estimated mathematically, and used to work out the best weight to give each verifier.

**Result:** this weighted combination clearly beat simple (unweighted) ensembling — with the biggest gains on the hardest datasets (like GPQA Diamond, MATH, and MMLU Pro), where the starting baseline accuracy was lower to begin with.


# Comparing Ways to Scale Verification Compute
There are several ways to spend more compute on verification: sample more candidate solutions, use bigger generator/verifier models, or add more verifiers to the ensemble pool.

## Comparing Methods
| Method | Description |
|---|---|
| **Pass@K Oracle** | Upper bound — assumes a perfect verifier always picks the correct answer |
| **Weaver Supervised** | Uses a large set of labeled data to learn verifier weights |
| **Weaver Unsupervised** | Uses only ~1% of the training labels |
| **Naive Ensemble** | Simple averaging of (already-filtered) verifiers, no learned weights |
| **Majority Voting** | Just picks the most common final answer |
| **Multi-Agent Verification (MAV)** | LLMs scoring responses using prompted rubrics |

**Result:** Weaver (both supervised and unsupervised versions) and even naive ensembling clearly beat majority voting and MAV. On hard problems, this pushed accuracy from just over 40% up to over 70% — enough to match a strong proprietary model like o3-mini.


# The Big Result: Small Models Can Match Bigger Ones
Using an 8B generator model together with a pool of 8B-and-under verifiers (combined via Weaver), accuracy reached about **70%** on these datasets — nearly matching what majority voting achieves with a **70B model**. In other words, smart verification let an 8B model perform roughly like a 70B model.

Applying the same approach to 70B-class models pushed accuracy to **86.2%** — very comparable to o3-mini, a much larger proprietary model.

Note: unlike coverage numbers discussed earlier, these results are about the **actual final accuracy of the whole system** (pass@1-style), not just "is a correct answer somewhere in the samples."


# Making Weaver Cheaper: Distillation
Running many separate verifier models for every sample is expensive — especially with 100+ samples per question. The fix: **train Weaver once, then distill it into one much smaller model.**

- Original ensemble: models in the 70B range.
- Distilled model: as small as **400 million parameters**.
- The distilled model captured **98.7% of the accuracy** of the full ensemble, while cutting verification compute by up to **99.97%**.

For context on how big a jump this kind of verification can produce: the paper notes their result is comparable to the well-known jump from GPT-4o to o3-mini (69.0% → 86.7% on a similar benchmark set) — a gain that normally requires heavy fine-tuning and post-training work, achieved here largely through smarter verification instead.

Both the original Weaver ensemble and the distilled version are open-sourced, with checkpoints available. Comparing efficiency (accuracy vs. total inference compute spent): the distilled version was far more efficient, and even the original (non-distilled) Weaver was more compute-efficient than naive ensembling or majority voting, simply because it reached higher accuracy for the same compute.


# Recap: Four Papers, One Storyline
1. **Training Verifiers to Solve Math Problems** — the first verifier: score a whole solution, train on human-labeled correct/incorrect data.
2. **Let's Verify Step by Step** — process reward models (PRMs) beat outcome reward models (ORMs), but need lots of human step-level labels.
3. **Math-Shepherd** — remove the need for human labels by estimating step quality through sampling ("does this step tend to lead somewhere correct?").
4. **Weaver** — instead of training one verifier, combine many existing imperfect verifiers with learned weights, and distill the result into a small, cheap model.

## Big Takeaways
- Verification helps both **at test time** (picking the best answer) and **during training** (as a reward signal for RL).
- **Process-based** rewards generally beat **outcome-based** ones — but combining both tends to work best in practice.
- Verifier quality keeps improving with more training data.
- Weaver shows a different lever entirely: scale verification by adding more *verifiers*, not just more *samples* from one verifier — and this can be made cheap through distillation.


# Closing Q&A

## Does Verification Work Outside Math? (e.g., Coding)
Yes — some of the benchmarks already included coding problems, and coding is a strong area for verification generally. A related approach, **CodeMonkeys** ([arxiv.org/abs/2501.14723](https://arxiv.org/abs/2501.14723)), skips training a dedicated verifier altogether: instead, the model generates its own **unit tests**, and those tests serve as the verifier.

## Does This Still Help "Reasoning" Models That Already Think Internally?
Yes, it still helps. Reasoning models already use something similar internally — they generate reasoning steps, get rewarded for good ones, and use RL to reinforce good reasoning "trajectories," which then get used as training data for the next round. So test-time scaling (like repeated sampling) is baked into how these models are built and trained — but it still helps them too, since more sampling still lets them explore more of the solution space.

## Will Repeated Sampling Eventually Just Become a Training Tool (and Not Needed at Test Time)?
This is the direction people hope for: a model so good at **pass@1** that you only need to ask it once. The challenge: pushing a model hard toward one "best" answer risks **losing creativity and diversity** in how it approaches problems — which is actually a valuable property models currently have. So there's a real tension between making pass@1 as strong as possible and preserving diverse thinking.

## Does It Matter If the Generator and Verifier Come From the Same Model Family?
An open research question without a clear settled answer yet. What is known: models tend to favor their **own** generations and their own way of interpreting results, compared to outputs from a different model family — this connects to related research on how some models benefit from being judged by a different class of verifier than themselves. But whether same-family-and-size vs. different-family-and-size specifically changes verifier performance hasn't been directly studied yet.
